### MODULES

In [1]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import ConcatDataset, DataLoader
torch.set_default_dtype(torch.float32)

In [2]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
setup_seed(888888)

### complete data

In [ ]:
class mydata():
    def __init__(self, folderpath,seqlength = 800,row = 100):
        self.path = folderpath
        self.length = seqlength
        self.row = row
        self.files = os.listdir(self.path) 
        self.files = sorted(self.files, key=lambda x: int(''.join(filter(lambda ch: ch.isdigit(), x.split('_')[1]))))
    def getdata(self):
        data = np.ones((1, self.length)) 
        labels = []                                       
        for i, file in enumerate(self.files): 
            file_path = os.path.join(self.path, file) 
            data_train = (pd.read_csv(file_path,header=None).values)
            data_train = data_train[0:self.row,-self.length:]  
            data = np.vstack((data, data_train))
            labels += [(0.4 + i*0.05)*1]* data_train.shape[0]  
        data = np.delete(data, 0, 0) 
        np.random.seed(888)
        idx = np.random.permutation(data.shape[0])   # random perm
        data = data[idx]
        labels = np.array(labels)[idx]
        data = torch.tensor(data, dtype=torch.float32)
        label = torch.tensor(labels, dtype = torch.float32)
        return  data, label

In [ ]:
class mydata2():
    def __init__(self, folderpath,seqlength = 800,row = 10,nlist = [0,1,2,3,4]):
        self.path = folderpath
        self.length = seqlength
        self.row = row 
        self.files = os.listdir(self.path) 
        self.n = nlist 
    def getdata(self):
        data = np.ones((1, self.length)) 
        labels = []                                       
        for i in self.n:
            file_path = os.path.join(self.path, self.files[i]) 
            data_train = (pd.read_csv(file_path,header=None).values)
            data_train = data_train[0:self.row,-self.length:]  
            data = np.vstack((data, data_train))
            labels += [(0.5 + i*0.5)*1]* data_train.shape[0]  
        data = np.delete(data, 0, 0) 
        np.random.seed(888)
        idx = np.random.permutation(data.shape[0])  # random perm
        data = data[idx]
        labels = np.array(labels)[idx]
        data = torch.tensor(data, dtype=torch.float32)
        label = torch.tensor(labels, dtype = torch.float32)
        return  data, label

In [5]:
class wavedataset(Dataset):
    def __init__(self, wavedata, wavelabel,seq):
        self.data = wavedata
        self.label = wavelabel
        self.seq = seq
    def __getitem__(self, idx):
        self.data = self.data.reshape(-1, 1, self.seq)  
        data = self.data[idx]
        label = self.label[idx]
        return data, label
    def __len__(self):
        return len(self.data)

In [6]:
def get_k_fold_data(k, i, data, label):
    assert k > 1
    fold_size = data.shape[0] // k
    data_train, label_train = [], []
    
    for j in range(k):
        idx = slice(j * fold_size, (j + 1) * fold_size)
        data_part, label_part = data[idx, :], label[idx]
        if j == i:
            data_valid, label_valid = data_part, label_part
        else:
            data_train.append(data_part)
            label_train.append(label_part)
    data_train = torch.cat(data_train, dim=0)
    label_train = torch.cat(label_train, dim=0)
    return data_train, label_train, data_valid, label_valid

### UDANN MODEL

In [7]:
from torch.autograd import Function
class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

In [8]:
class cnn(nn.Module):
    def __init__(self):
        super(cnn, self).__init__()
        self.feature = nn.Sequential(
                nn.Conv1d(1,16,kernel_size = 5,padding = 2),
                nn.MaxPool1d(kernel_size = 2),
                nn.BatchNorm1d(16),
                nn.ReLU(True),
                nn.Conv1d(16,32,kernel_size= 5, padding = 2),
                nn.MaxPool1d(kernel_size = 2),
                nn.BatchNorm1d(32),
                nn.ReLU(True),
                nn.Conv1d(32,32,kernel_size= 5, padding = 2),
                nn.MaxPool1d(kernel_size = 2),
                nn.ReLU(True),
                nn.Conv1d(32,8,kernel_size= 3,padding = 1,stride = 2),
                nn.BatchNorm1d(8),
                nn.MaxPool1d(kernel_size = 2),             
                nn.Flatten())
        self.reg = nn.Sequential(
                nn.Linear(200,10),
                nn.ReLU(True),
                nn.Linear(10,1),
                nn.ReLU(True),
                )                                               
        self.domain_classifier =nn.Sequential(
                nn.Linear(200,10),
                nn.ReLU(True),
                nn.Linear(10,2),
                nn.ReLU(True),
                )   
        
    def forward(self,input_data,alpha=1):
        feature = self.feature(input_data)
        reverse_feature = ReverseLayerF.apply(feature,alpha)
        x = self.reg(feature)
        d = self.domain_classifier(reverse_feature)
        return x,d,feature

In [9]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_elements = y_true != 0
    mape = np.mean(np.abs((y_true[non_zero_elements] - y_pred[non_zero_elements]) / y_true[non_zero_elements])) * 100
    return mape

In [10]:
def acc(y_pred, y_true):
    y_pred_labels = torch.argmax(y_pred, dim=1)
    correct_predictions = torch.sum(y_pred_labels == y_true).item()  
    accuracy = correct_predictions / y_true.size(0)
    return accuracy

### TRAIN VALID

##### $dataset$

In [ ]:
''' 1 = simulation dataset; 2 = experimental dataset; sf = shuffle; v = valid dataset ; t = test dataset'''
seq = 800
row1 = 100 
path = "E:/School/CODE/FEMassistedNN/nsysim/snr9"  
# random order complete sim data in path 
DATA = mydata(path,seq,row1)  
data1,label1 = DATA.getdata()  # got 4500 simulation samples
label1 = label1.reshape(-1, 1)

n = [0,1,2,3,4];# five actual diameters index
m = len(n)
row2 = 10
path2 = "E:/School/CODE/FEMassistedNN/nsyexp/snr9"  
# random order complete exp data in path2 
DATA2 = mydata2(path2,seq,row2,n) 
data2,label2 = DATA2.getdata()  # got 50 actual experimental samples 
label2 = label2.reshape(-1, 1)

data3 = torch.zeros(row2*m,seq) # got 50 paired samples for UDANN
label3 = torch.zeros(row2*m,1)  # got 50 paired samples for UDANN
for i in range(m):
    start_idx = row1 * 2 + 10 * row1 * i
    end_idx = start_idx + row2
    data3[i * row2: i * row2+row2, :] = data1[start_idx:end_idx, :].clone()
    label3[i * row2: i * row2+row2, :] = label1[start_idx:end_idx, :].clone()

'''diamter labels normalization'''
label11 = (label1-min(label1))/(max(label1)-min(label1))+0.00001
label21 = (label2-min(label1))/(max(label1)-min(label1))
label31 = (label3-min(label1))/(max(label1)-min(label1))

In [ ]:
'''sf = shuffle ; v = valid ; t = test'''
random.seed(88888)
index = [i for i in range(len(data1))] 
random.shuffle(index)
data1sf = data1[index]   
label1sf= label11[index]
data1t,label1t,data1v,label1v = get_k_fold_data(5, 1, data1sf, label1sf)
# data1t,label1t,3600 sim samples for train (feature extractor and regression net)
# data1v,label1v,900 sim samples for valid "regression net"

In [ ]:
# paired samples
random.seed(88888)
index = [i for i in range(len(data2))] 
random.shuffle(index)
data2sf = data2[index]   
# 50 exp samples for train (feature extractor and domain classifier) valid "regression net" 
label2sf = label21[index]  
# 50 exp samples for train (feature extractor and domain classifier) valid "regression net"

random.seed(888888)
data3sf = data3[index]  
# 50 sim samples for train (feature extractor and domain classifier)
label3sf = label31[index]  
# 50 sim samples for train (feature extractor and domain classifier) 
# data2sf,data3sf are paired to train (feature extractor and domain classifier) 

##### $dataloader$

In [15]:
batchsize1 = 40
batchsize2 = 10
dataset_sourcebig = wavedataset(data1t,label1t,seq)
dataloader_sourcebig = DataLoader(dataset=dataset_sourcebig,
    batch_size=batchsize1,shuffle=False,num_workers=0)
dataset_target = wavedataset(data2sf,label2sf,seq)
dataloader_target = DataLoader(dataset=dataset_target,
    batch_size=batchsize2,shuffle=False,num_workers=0)
dataset_sourcesmall = wavedataset(data3sf,label3sf,seq)
dataloader_sourcesmall = DataLoader(dataset=dataset_sourcesmall,
    batch_size=batchsize2,shuffle=False,num_workers=0)

In [16]:
len_sourcebig = len(dataloader_sourcebig)
len_target = len(dataloader_target)
len_sourcesmall = len(dataloader_sourcesmall)

repeat_target = math.ceil(len_sourcebig / len_target)
print(repeat_target)
repeat_sourcesmall = math.ceil(len_sourcebig / len_sourcesmall)
print(repeat_sourcesmall)

extended_dataset_target = ConcatDataset([dataset_target] * repeat_target)
extended_dataset_sourcesmall = ConcatDataset([dataset_sourcesmall] * repeat_sourcesmall)

dataloader_target = DataLoader(dataset=extended_dataset_target,
    batch_size=batchsize2, shuffle=False, num_workers=0)
dataloader_sourcesmall = DataLoader(dataset=extended_dataset_sourcesmall,
    batch_size=batchsize2, shuffle=False, num_workers=0)

len_dataloader_target = len(dataloader_target)
len_dataloader_sourcesmall = len(dataloader_sourcesmall)

print(f"Sourcebig batch count: {len_sourcebig}")
print(f"Extended Target batch count: {len_dataloader_target}")
print(f"Extended Sourcesmall batch count: {len_dataloader_sourcesmall}")
len_dataloader = min(len_sourcebig, len_dataloader_target,len_dataloader_sourcesmall)
print(len_dataloader)

18
18
Sourcebig batch count: 90
Extended Target batch count: 90
Extended Sourcesmall batch count: 90
90


##### $train$

In [17]:
loss_mae = nn.L1Loss()
loss_domain = nn.CrossEntropyLoss()
''' experimental = target = t; simulation = source = s '''
r2_t,r2_st,r2_sv = [],[],[]
mse_t,mse_st,mse_sv = [],[],[]
mae_t,mae_st,mae_sv = [],[],[]
mape_t,mape_st,mape_sv = [],[],[]
loss_f,loss_dcre,loss_all = [],[],[]

In [23]:
lr = 0.00006
net = cnn()
opt = optim.Adam(net.parameters(), lr = lr)
n_epoch = 200
ld=0.05
lf=0.2
best1 = 15; best2 = 0.1

path = 'D:/OneDrive/Code/master/DANN_2601/DANN/'
modelname = f'UDANN2604-bt1-{batchsize1}bt2-{batchsize2}_lr{lr}_e{n_epoch}_row{row2}_ld{ld}_lf{lf}'
dataset = '-snr9'
modelpath = path + modelname + dataset+'.pth'
print(modelpath)

D:/OneDrive/Code/master/DANN_2601/DANN/UDANN2604-bt1-40bt2-10_lr6e-05_e200_row10_ld0.05_lf0.2-snr9.pth


In [ ]:
for epoch in range(n_epoch):
    
    net.train()
    sourcebig_iter = iter(dataloader_sourcebig)
    sourcesmall_iter = iter(dataloader_sourcesmall)
    target_iter = iter(dataloader_target)
    for i in range(len_dataloader):
        opt.zero_grad()
        p = float(i + epoch * len_dataloader) / n_epoch / len_dataloader
        a = 2. / (1. + np.exp(-10 * p)) - 1

        s_data,s_reg_label = next(sourcebig_iter); # 3600 sim samples for train (feature extractor and regression net)
        s_reg_label = s_reg_label.reshape(batchsize1,1)
        s_reg,_,_ = net(s_data,alpha = a)
        loss_mae_s = loss_mae(s_reg,s_reg_label)  # regression loss (sim)
        
        s_data,s_reg_label = next(sourcesmall_iter)  
        # 50 paired sim samples for train (feature extractor and domain classifier)
        s_domain_label = torch.zeros(batchsize2).long()
        _, s_domain,s_feature = net(s_data,alpha = a)
        loss_domain_s = loss_domain(s_domain,s_domain_label) # classfication loss1 (sim)
        t_data,_ = next(target_iter)  
        # 50 paired exp samples for train (feature extractor and domain classifier)
        t_domain_label = torch.ones(batchsize2).long()
        _, t_domain, t_feature = net(t_data,alpha = a) 
        loss_domain_t = loss_domain(t_domain,t_domain_label) #classfication loss2 (exp)
        lossdomaincre = loss_domain_s + loss_domain_t  # classification loss

        loss_feature = loss_mae((s_feature),(t_feature))  # feature loss (sim-exp)
        
        loss = loss_mae_s + ld*lossdomaincre + lf*loss_feature  # total loss
        loss.backward()
        opt.step() 

    net.eval()
    with torch.no_grad():
        s_label = label1v.numpy()
        s_data =(data1v.reshape(-1, 1, seq)) # 900 sim samples for valid "regression net" 
        s_labelpred,s_dpred,_ = net(s_data,alpha=a) 
        s_dtrue = torch.zeros(s_dpred.size(0))
        s_labelpred = s_labelpred.numpy()

        t_label=label2sf.numpy()  
        t_data = (data2sf.reshape(-1, 1, seq)) # 50 exp samples for valid "regression net" 
        t_labelpred,t_dpred,_ = net(t_data,alpha = a);t_dtrue = torch.ones(t_dpred.size(0))  
        t_labelpred = t_labelpred.numpy()
        
    min_label1 = torch.min(label1).item()
    max_label1 = torch.max(label1).item()
    s_label = s_label * (max_label1 - min_label1) + min_label1
    s_labelpred = s_labelpred * (max_label1 - min_label1) + min_label1
    t_label = t_label * (max_label1 - min_label1) + min_label1
    t_labelpred = t_labelpred * (max_label1 - min_label1) + min_label1
    
    s_mse = mean_squared_error(s_label,s_labelpred)
    s_mae = mean_absolute_error(s_label,s_labelpred)
    s_mape = mape(s_label,s_labelpred)
    s_r2 = r2_score(s_label,s_labelpred)
    s_acc = acc(s_dpred,s_dtrue)

    t_mse = mean_squared_error(t_label,t_labelpred)
    t_mae = mean_absolute_error(t_label,t_labelpred)
    t_mape = mape(t_label,t_labelpred)
    t_r2 = r2_score(t_label,t_labelpred)
    t_acc = acc(t_dpred,t_dtrue)
    
    r2_t.append(t_r2);mse_t.append(t_mse);
    mae_t.append(t_mae);mape_t.append(t_mape);
    r2_sv.append(s_r2);mse_sv.append(s_mse);
    mae_sv.append(s_mae);mape_sv.append(s_mape);
    
    loss_f.append(loss_feature.item());
    loss_dcre.append(lossdomaincre.item());
    loss_all.append(loss.item())
    
    print(f'Epoch {epoch + 1}')
    print(f'    loss_regression：{loss_mae_s.item():.10f}')
    print(f'    loss_classfication：{lossdomaincre.item():.10}')
    print(f'    loss_feature:{loss_feature.item():.10f}')
    print(f'    loss_all:{loss.item():.10f}，lf:{lf}，ld:{ld}')
    print(f'    acc_sim:{s_acc:.5f},acc_exp:{t_acc:.5f}')
    print(f'    MAE(T):{t_mae:.4f},MAPE(T):{t_mape:.2f}%')
    print(f'    MAE(Sv):{s_mae:.4f},MAPE(Sv):{s_mape:.2f}%')
    
    if t_mape < best1:
        path1 = path + modelname + dataset +'-1.pth'
        torch.save(net,path1)
        best1 = t_mape
        print('  ***save1***')
    if t_mae < best2:
        path2 = path + modelname + dataset +'-2.pth'
        torch.save(net,path2)
        best2 = t_mae
        print('  ***save2***')
    print(f'    best MAPE:{best1}，best MAE:{best2}')

Epoch 1
    loss_regression：0.0634820238
    loss_classfication：1.378748894
    loss_feature:0.3207943439
    loss_all:0.1965783387，lf:0.2，ld:0.05
    acc_sim:0.81333,acc_exp:0.30000
    MAE(T):0.3913,MAPE(T):35.68%
    MAE(Sv):0.1121,MAPE(Sv):7.96%
    best MAPE:15，best MAE:0.1
Epoch 2
    loss_regression：0.0266152509
    loss_classfication：1.316627264
    loss_feature:0.2441929877
    loss_all:0.1412852108，lf:0.2，ld:0.05
    acc_sim:0.64667,acc_exp:0.60000
    MAE(T):0.3840,MAPE(T):28.65%
    MAE(Sv):0.0592,MAPE(Sv):4.53%
    best MAPE:15，best MAE:0.1
Epoch 3
    loss_regression：0.0187882148
    loss_classfication：1.275952578
    loss_feature:0.2112263143
    loss_all:0.1248311102，lf:0.2，ld:0.05
    acc_sim:0.63000,acc_exp:0.60000
    MAE(T):0.3875,MAPE(T):26.45%
    MAE(Sv):0.0434,MAPE(Sv):3.44%
    best MAPE:15，best MAE:0.1
Epoch 4
    loss_regression：0.0161259044
    loss_classfication：1.250089884
    loss_feature:0.1963464171
    loss_all:0.1178996786，lf:0.2，ld:0.05
    acc_sim:0

train performance is different from indepentdent test set

In [24]:
modelpath1 = modelname+ dataset +'last.pth'
torch.save(net,modelpath1)
print(modelpath1)

UDANN2604-bt1-40bt2-10_lr6e-05_e200_row10_ld0.05_lf0.2-snr9last.pth


#### SAVE LOSS

In [21]:
lists = {
    'r2_t': r2_t,
    'r2_sv': r2_sv,
    'mse_t': mse_t,
    'mse_sv': mse_sv,
    'mae_t': mae_t,
    'mae_sv': mae_sv,
    'mape_t': mape_t,
    'mape_sv': mape_sv,
    'loss_dcre':loss_dcre,
    'loss_fmae':loss_f,
    'loss_all':loss_all,
}
df = pd.DataFrame(lists)
df.to_csv(modelname+dataset+'-1.csv', index=False)
print('down')

down
